## Código dado

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__ if hasattr(keras, "__version__") else "tf.keras")

TensorFlow: 2.20.0
Keras: 3.13.2


## Dataset CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
class_names = np.array([
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
])


x_train = x_train[:1000].astype("float32")/255.0
y_train = y_train[:1000].astype("float32")/255.0

x_test = x_test[:1000].astype("float32")/255.0
y_test = y_test[:1000].astype("float32")/255.0


print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
(1000, 32, 32, 3) (1000, 1)
(1000, 32, 32, 3) (1000, 1)


In [ ]:
def make_class(images, labels, batch_size = 64, training=True):
  ds = tf.data.Dataset.from_tensor_slices((images, labels))
  if training :
    ds = ds.shuffle(4*batch_size)

  return ds

train_ds = make_class(x_train, y_train, training=True)
test_ds = make_class(x_test, y_test, training=False)



---->
entrada  ||| [bloque1] [bloque2] [bloque3] [bloque4]  |||  [bloque1] [bloque2] [bloque3] [bloque4]
       _________________________| __________________|~~~~~~~~~~~~~~~~~~~~~~~~~|




# **Arquitectura resnet a mano**


## Definimos los bloques

In [ ]:
def conv_relu(x, filters, kernel_size=3, stride=1):
    x = layers.Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    x=layers.BatchNormalization()(x)
    x=layers.Activation("relu")(x)

    return x

def resid_block(x, filters, stride, projection):
    x_atajo = x
    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        kernel_initializer="he_normal"
    )(x)
    x=layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)


    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
    )(x)
    x=layers.BatchNormalization()(x)

    if projection:
        x_atajo = layers.Conv2D(
          filters=filters,
          kernel_size=1,
          strides=stride,
          padding="same",
          kernel_initializer="he_normal"
        )(x_atajo)
        x_atajo = layers.BatchNormalization()(x_atajo)


    x=layers.Add()([x, x_atajo])
    x=layers.Activation("relu")(x)

    return x





## ResNet pequeña

3 bloques con 16f -> 3 bloques 32f/2-> 3 bloques 64f/2 -> GlobalAveragePooling2D -> Dense(10)

In [ ]:
def build_resnet(input_shape, num_classes, filter, blocks):
  input = keras.Input(shape=input_shape)
  x = conv_relu(input, filters=filter[0], kernel_size=3, stride=1)


  for indice_stage, (filters, n_blocks) in enumerate(zip(filter, blocks), start=0):
    for indice_bloque in range(n_blocks):
      is_first_block = indice_bloque==0
      is_first_block_network = indice_stage==0



      stride = 2 if indice_stage > 0 and indice_bloque == 0 else 1
      use_projection = stride != 1 or x.shape[-1] != filters


      x = resid_block(x, filters, stride, use_projection)


  x = layers.GlobalAveragePooling2D()(x)
  outputs = layers.Dense(num_classes, activation="softmax")(x)

  return keras.Model(inputs=input, outputs=outputs, name="resnet_cifar_manual")


nuestra_red_resnet = build_resnet((32,32,3), 10, (16,32,64), (3,3,3))
nuestra_red_resnet.summary()


Model: "resnet_cifar_manual"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_187 (Conv2D) │ (None, 32, 32,    │        448 │ input_layer_14[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_187[0][0]  │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_156      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_188 (Conv2D) │ (None, 32, 32,    │      2,320 │ activation_156[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_188[0][0]  │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_157      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_189 (Conv2D) │ (None, 32, 32,    │      2,304 │ activation_157[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_189[0][0]  │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_79 (Add)        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 16)               │            │ activation_156[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_158      │ (None, 32, 32,    │          0 │ add_79[0][0]      │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_190 (Conv2D) │ (None, 32, 32,    │      2,320 │ activation_158[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_190[0][0]  │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_159      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_191 (Conv2D) │ (None, 32, 32,    │      2,304 │ activation_159[0… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │         64 │ conv2d_191[0][0]  │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_80 (Add)        │ (None, 32, 32,    │          0 │ batch_normalizat

 Total params: 274,490 (1.05 MB)

 Trainable params: 272,922 (1.04 MB)

 Non-trainable params: 1,568 (6.12 KB)

## Resnet keras

# Transfer learning con ResNet50

## Preparar datos para ResNet50

## Transfer learning

Feature Extraction

## Fine tuning

## Comparación final